# Prelude

In [1]:
%cd ~/TP/connections

/home/mfixman/Dropbox/continuous/TP/connections


In [2]:
import numpy
import scipy
import matplotlib
import seaborn
import pandas
import os
import sys

from matplotlib import pyplot

# Getting Data

## Read CSV

In [3]:
prover9 = pandas.read_csv(
    'external_results/prover9.csv',
)

In [4]:
prover9.head()

,problem,part,declared_status,strategy,result,time_ms,steps,proved,clauses,score_ms,generated,kept,error,score_error
0,ALG004-1.p,0,Unsatisfiable,WeightedSupport16,Proved,182.510,42.0,True,7.0,NaN,4733.0,1221.0,NaN,NaN
1,ALG004-1.p,0,Unsatisfiable,Baseline,Proved,174.269,42.0,True,7.0,NaN,4733.0,1221.0,NaN,NaN
2,ALG004-1.p,0,Unsatisfiable,ResumeSupport1,Proved,173.144,42.0,True,7.0,NaN,4733.0,1221.0,NaN,NaN
3,ALG069-10.p,0,Unsatisfiable,WeightedSupport16,Proved,32.997,25.0,True,14.0,NaN,121.0,69.0,NaN,NaN
4,ALG069-10.p,0,Unsatisfiable,ResumeSupport1,Proved,28.024,25.0,True,14.0,NaN,121.0,69.0,NaN,NaN


## Errors

In [5]:
prover9.loc[lambda x: x.result == 'Error'].error.iloc[0]

'ValueError("Unsupported symbol \'E\'; numeric and distinct objects need explicit encoding.")'

In [6]:
prover9.loc[lambda x: x.result == 'Error'].problem.str[:3].value_counts()

problem
SYO    150
NUN      6
LCL      6
ANA      3
SYN      3
Name: count, dtype: int64

In [7]:
prover9.loc[lambda x: x.result == 'Error'].loc[lambda x: x.problem.str[:3] != 'SYO'].error

4851     ValueError("Unsupported symbol '+'; numeric an...
4852     ValueError("Unsupported symbol '+'; numeric an...
4853     ValueError("Unsupported symbol '+'; numeric an...
7614     ValueError("Unsupported symbol '+'; numeric an...
7615     ValueError("Unsupported symbol '+'; numeric an...
7616     ValueError("Unsupported symbol '+'; numeric an...
8646     ValueError("Unsupported symbol '+'; numeric an...
8647     ValueError("Unsupported symbol '+'; numeric an...
8648     ValueError("Unsupported symbol '+'; numeric an...
12324    ValueError("Unsupported symbol '+'; numeric an...
12325    ValueError("Unsupported symbol '+'; numeric an...
12326    ValueError("Unsupported symbol '+'; numeric an...
16185    ValueError("Unsupported symbol '+'; numeric an...
16186    ValueError("Unsupported symbol '+'; numeric an...
16187    ValueError("Unsupported symbol '+'; numeric an...
16878    ValueError("Unsupported symbol 'A proposition'...
16879    ValueError("Unsupported symbol 'A proposition'.

## Filtering

In [8]:
prover9 = prover9.loc[lambda x: x.result != 'Error']

# Analysis

In [9]:
prover9.groupby('strategy').result.value_counts().to_frame()

count
strategy          result         
Baseline          Proved     3825
                  Unproved   2034
                  Timeout     354
ResumeSupport1    Proved     3921
                  Unproved   1870
                  Timeout     422
WeightedSupport16 Proved     3812
                  Unproved   2035
                  Timeout     366

In [10]:
merged = pandas.merge(
    prover9,
    prover9,
    left_on = ['problem', 'part', 'declared_status'],
    right_on = ['problem', 'part', 'declared_status'],
    suffixes = ('_left', '_right'),
)

merged.head()

,problem,part,declared_status,strategy_left,result_left,time_ms_left,steps_left,proved_left,clauses_left,score_ms_left,...,result_right,time_ms_right,steps_right,proved_right,clauses_right,score_ms_right,generated_right,kept_right,error_right,score_error_right
0,ALG004-1.p,0,Unsatisfiable,WeightedSupport16,Proved,182.510,42.0,True,7.0,NaN,...,Proved,182.510,42.0,True,7.0,NaN,4733.0,1221.0,NaN,NaN
1,ALG004-1.p,0,Unsatisfiable,WeightedSupport16,Proved,182.510,42.0,True,7.0,NaN,...,Proved,174.269,42.0,True,7.0,NaN,4733.0,1221.0,NaN,NaN
2,ALG004-1.p,0,Unsatisfiable,WeightedSupport16,Proved,182.510,42.0,True,7.0,NaN,...,Proved,173.144,42.0,True,7.0,NaN,4733.0,1221.0,NaN,NaN
3,ALG004-1.p,0,Unsatisfiable,Baseline,Proved,174.269,42.0,True,7.0,NaN,...,Proved,182.510,42.0,True,7.0,NaN,4733.0,1221.0,NaN,NaN
4,ALG004-1.p,0,Unsatisfiable,Baseline,Proved,174.269,42.0,True,7.0,NaN,...,Proved,174.269,42.0,True,7.0,NaN,4733.0,1221.0,NaN,NaN


In [11]:
better = merged.loc[lambda x: x.proved_left & ~x.proved_right]

In [12]:
pandas.crosstab(better.strategy_left, better.strategy_right)

strategy_right,Baseline,ResumeSupport1,WeightedSupport16
strategy_left,,,
Baseline,0,46,110
ResumeSupport1,142,0,173
WeightedSupport16,97,64,0


In [13]:
pandas.crosstab(better.strategy_left, better.strategy_right).div(prover9.problem.nunique()).style.format('{:.2%}')

strategy_right,Baseline,ResumeSupport1,WeightedSupport16
strategy_left,,,
Baseline,0.00%,0.74%,1.77%
ResumeSupport1,2.29%,0.00%,2.78%
WeightedSupport16,1.56%,1.03%,0.00%


In [15]:
test = better.loc[lambda x: x.part == 9]
pandas.crosstab(
    test.strategy_left,
    test.strategy_right
).div(
    prover9.loc[lambda x: x.part == 9].problem.nunique()
).style.format('{:.2%}')

strategy_right,Baseline,ResumeSupport1,WeightedSupport16
strategy_left,,,
Baseline,0.00%,0.52%,1.73%
ResumeSupport1,2.94%,0.00%,2.76%
WeightedSupport16,2.07%,0.69%,0.00%
